# L1 and L2 regularisation in a neural network
Why a bigger network overfits more, then the same 128-128 network on `make_moons` without regularisation,
with L2 and with L1: decision boundaries, training curves and the first-layer weights.
Results go to `data/` for the figures in `images/`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_moons

tf.config.experimental.enable_op_determinism()   # same numbers on every run
os.makedirs("data", exist_ok=True)

## The data
100 points in two noisy half-moons.

In [ ]:
X, y = make_moons(n_samples=100, noise=0.25, random_state=2)
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y}).round(4).to_csv("data/points.csv", index=False)
xx, yy = np.meshgrid(np.linspace(-2, 3, 151), np.linspace(-1.75, 2.25, 121))
grid = np.c_[xx.ravel(), yy.ravel()]
boundaries = {"x1": grid[:, 0], "x2": grid[:, 1]}

## 1. More neurons, more complex boundaries
One hidden layer with 1, 10, 50 or 1,000 ReLU neurons, trained on all 100 points.

In [ ]:
for n in [1, 10, 50, 1000]:
    keras.utils.set_random_seed(0)
    model = keras.Sequential([keras.Input(shape=(2,)),
                              keras.layers.Dense(n, activation="relu"),
                              keras.layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy",
                  metrics=["accuracy"])
    model.fit(X, y, epochs=1000, verbose=0)
    _, acc = model.evaluate(X, y, verbose=0)
    print(f"{n:5d} neurons: training accuracy {acc:.2f}")
    boundaries[f"n={n}"] = model.predict(grid, verbose=0).ravel()

### Each neuron's hyperplane and the straight segments of the decision boundary


In [ ]:
# Each hidden neuron's hyperplane, and how many straight segments the decision boundary has.
# Same four networks as above (same seed and settings), keeping their weights for images/neuron_lines.py.
xx2, yy2 = np.meshgrid(np.linspace(-2, 3, 501), np.linspace(-1.75, 2.25, 401))
grid2 = np.c_[xx2.ravel(), yy2.ravel()]
rows, summary, preds = [], [], {}
for n in [1, 10, 50, 1000]:
    keras.utils.set_random_seed(0)
    model = keras.Sequential([keras.Input(shape=(2,)),
                              keras.layers.Dense(n, activation="relu"),
                              keras.layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy",
                  metrics=["accuracy"])
    model.fit(X, y, epochs=1000, verbose=0)
    W, b, V, c = model.get_weights()
    rows += [dict(n=n, i=i, w1=W[0, i], w2=W[1, i], b=b[i], v=V[i, 0]) for i in range(n)]
    rows.append(dict(n=n, i=-1, w1=0.0, w2=0.0, b=c[0], v=0.0))          # i = -1: the output bias
    H = grid2 @ W + b
    s = ((np.maximum(H, 0) @ V[:, 0] + c[0]) > 0).reshape(xx2.shape)
    A = (H > 0).reshape(xx2.shape + (n,))
    edge = np.zeros_like(s)
    edge[:, 1:] |= s[:, 1:] != s[:, :-1]
    edge[1:, :] |= s[1:, :] != s[:-1, :]
    pieces = len({A[r, q].tobytes() for r, q in zip(*np.nonzero(edge))})   # one segment per active-neuron pattern
    preds[n] = model.predict(X, verbose=0).ravel() > 0.5
    summary.append(dict(n=n, train_acc=float(np.mean(preds[n] == y)), straight_pieces=pieces))
pd.DataFrame(rows).round(5).to_csv("data/neuron_lines.csv", index=False)
pd.DataFrame(summary).to_csv("data/neuron_lines_summary.csv", index=False)
caught = (preds[1000] == y) & (preds[50] != y)        # fitted only by the 1,000-neuron network
pd.DataFrame({"x1": X[caught, 0], "x2": X[caught, 1], "y": y[caught]}).round(4).to_csv("data/pocket_points.csv", index=False)
print(pd.DataFrame(summary).to_string(index=False))
print("points caught only by the 1,000-neuron network:", int(caught.sum()))

## 2. The same network without and with regularisation

In [ ]:
def build(regularizer=None, seed=0):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

build().summary()

In [ ]:
from keras import regularizers

runs = {"none": None, "L2": regularizers.L2(0.03), "L1": regularizers.L1(0.001)}
histories, weights, scores = {}, {}, []
for name, reg in runs.items():
    model = build(reg)
    h = model.fit(X, y, epochs=2000, validation_split=0.2, verbose=0).history   # last 20 points validate
    histories[f"loss {name}"], histories[f"val_loss {name}"] = h["loss"], h["val_loss"]
    histories[f"acc {name}"], histories[f"val_acc {name}"] = h["accuracy"], h["val_accuracy"]
    W1 = model.get_weights()[0]                  # first layer: shape (2, 128)
    weights[name] = W1.reshape(256)
    scores.append({"regularisation": name, "train_acc": h["accuracy"][-1], "val_acc": h["val_accuracy"][-1],
                   "val_loss": h["val_loss"][-1], "w_min": W1.min(), "w_max": W1.max(),
                   "share_near_0": np.mean(np.abs(W1) < 1e-3)})
    boundaries[name] = model.predict(grid, verbose=0).ravel()
scores = pd.DataFrame(scores)
print(scores.round(3))

## 3. The penalty Keras adds
`L2(0.03)` adds 0.03 times the sum of squared weights of the layer to the loss (no 1/2, no 1/n).

In [ ]:
W = np.array([[0.5, -1.0], [2.0, 0.1]], dtype="float32")
print("L2(0.03):", float(regularizers.L2(0.03)(W)), "= 0.03 *", float((W ** 2).sum()))
print("L1(0.001):", float(regularizers.L1(0.001)(W)), "= 0.001 *", float(np.abs(W).sum()))

## 4. Sparse or only small? L1 against L2 with plain SGD
Same data and network, plain SGD (learning rate 0.01), 2,000 epochs, 3 seeds. Only the penalty changes. A weight counts as 0 when it is 0 to three decimal places.

In [ ]:
def build_sgd(regularizer, seed):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01),   # plain SGD
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

sparse_runs = {"L2": lambda: regularizers.L2(0.03), "L1": lambda: regularizers.L1(0.003)}
sparse_rows, sparse_w = [], {}
for name, make_reg in sparse_runs.items():
    for seed in range(3):                                  # average over 3 starting points
        m = build_sgd(make_reg(), seed)
        h = m.fit(X, y, epochs=2000, validation_split=0.2, verbose=0).history
        w = m.get_weights()[0].reshape(256)               # first-layer weights
        if seed == 0:
            sparse_w[name] = w
        sparse_rows.append({"penalty": name, "seed": seed, "zero_to_3_decimals": np.mean(np.abs(w) < 0.0005),
                            "largest": np.abs(w).max(), "val_acc": h["val_accuracy"][-1]})
        print(sparse_rows[-1], flush=True)
sparse = pd.DataFrame(sparse_rows)
print(sparse.groupby("penalty")[["zero_to_3_decimals", "largest", "val_acc"]].mean().round(3))
pd.DataFrame(sparse_w).round(6).to_csv("data/sparsity_weights.csv", index=False)
sparse.round(5).to_csv("data/sparsity.csv", index=False)


## 5. Supporting check: L1 and L2, crossed with SGD and Adam
Same data, network, seed and 2,000 epochs; only the optimizer and the penalty change. For each run: the share of first-layer weights below 0.001 and 0.01 in size, exact zeros, and the share of weights whose sign flipped in the last 100 epochs (a weight hovering around 0).

In [ ]:
# Run on CPU (CUDA_VISIBLE_DEVICES=""); about 10 minutes.
class Track(keras.callbacks.Callback):
    """Keeps the first-layer weights of the last 100 epochs."""
    def on_train_begin(self, logs=None): self.W = []
    def on_epoch_end(self, epoch, logs=None):
        if epoch >= 1900: self.W.append(self.model.get_weights()[0].ravel().copy())

opts = {"SGD 0.01": lambda: keras.optimizers.SGD(learning_rate=0.01),
        "SGD 0.1": lambda: keras.optimizers.SGD(learning_rate=0.1),
        "Adam 0.01": lambda: keras.optimizers.Adam(learning_rate=0.01)}
regs = {"none": lambda: None, "L2(0.03)": lambda: regularizers.L2(0.03), "L1(0.001)": lambda: regularizers.L1(0.001)}
rows = []
for o, mk_opt in opts.items():
    for r, mk_reg in regs.items():
        keras.utils.set_random_seed(0)
        model = keras.Sequential([keras.Input(shape=(2,)),
            keras.layers.Dense(128, activation="relu", kernel_regularizer=mk_reg()),
            keras.layers.Dense(128, activation="relu", kernel_regularizer=mk_reg()),
            keras.layers.Dense(1, activation="sigmoid")])
        model.compile(optimizer=mk_opt(), loss="binary_crossentropy", metrics=["accuracy"])
        t = Track()
        h = model.fit(X, y, epochs=2000, validation_split=0.2, verbose=0, callbacks=[t]).history
        W = np.abs(model.get_weights()[0])
        T = np.array(t.W)                                        # (100, 256)
        flips = (np.diff(np.sign(T), axis=0) != 0).any(axis=0)  # sign changed in the last 100 epochs
        rows.append({"optimizer": o, "penalty": r, "below_1e-3": np.mean(W < 1e-3),
                     "below_1e-2": np.mean(W < 1e-2), "exactly_0": int((W == 0).sum()),
                     "median_abs": np.median(W), "sign_flip_last100": flips.mean(),
                     "train_acc": h["accuracy"][-1], "val_loss": h["val_loss"][-1]})
        print(rows[-1], flush=True)
df = pd.DataFrame(rows)
print(df.round(4).to_string())
df.round(5).to_csv("data/optimizer_penalty.csv", index=False)


In [ ]:
# first-layer nodes whose two weights are both below 0.001 in size, in the runs of section 2
for name in ["none", "L2", "L1"]:
    W = np.abs(weights[name]).reshape(2, 128)
    print(name, (W.max(axis=0) < 1e-3).sum(), "of 128 nodes")


## Saving results for the figures

In [ ]:
pd.DataFrame(boundaries).round(4).to_csv("data/boundary_grid.csv", index=False)
pd.DataFrame(histories).round(5).to_csv("data/histories.csv", index_label="epoch")
pd.DataFrame(weights).round(5).to_csv("data/first_layer_weights.csv", index=False)
scores.round(5).to_csv("data/scores.csv", index=False)
print("saved")

## Snapshots of the first-layer weights during training
The runs of section 7.2 and 7.3 again (same starting weights), keeping the 256 first-layer weights at selected epochs. Run on CPU.

In [ ]:
SNAP = [0, 1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 70, 100, 150, 200, 300, 500, 700, 1000, 1500, 2000]

class Snap(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.W = {0: self.model.get_weights()[0].ravel().copy()}
    def on_epoch_end(self, epoch, logs=None):
        if epoch + 1 in SNAP: self.W[epoch + 1] = self.model.get_weights()[0].ravel().copy()

snap_rows = []
for name in ["none", "L2"]:
    model, s = build(regularizers.L2(0.03) if name == "L2" else None), Snap()
    model.fit(X, y, epochs=2000, validation_split=0.2, verbose=0, callbacks=[s])
    print(name, "final range", s.W[2000].min().round(2), s.W[2000].max().round(2))   # section 7.4 table
    for e, w in s.W.items():
        snap_rows += [{"run": name, "epoch": e, "w": v} for v in w]
pd.DataFrame(snap_rows).round(5).to_csv("data/weight_snapshots.csv", index=False)

## 6. How strong should the penalty be? A sweep over λ
The network of section 2 (same seed, same data) trained with eight values of λ, from 0 to 1. For each: the decision surface, the scores, the first-layer weights, and the **sensitivity**: how much the predicted probability changes, on average, when a training point is moved by 0.1 along either feature. Run on CPU.

In [ ]:
LAMS = [0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
cx, cy = np.meshgrid(np.linspace(-2, 3, 76), np.linspace(-1.75, 2.25, 61))      # a coarser grid keeps the file small
coarse = np.c_[cx.ravel(), cy.ravel()]
sweep_rows, sweep_w, sweep_grid = [], {}, {"x1": coarse[:, 0], "x2": coarse[:, 1]}
moves = np.array([[0.1, 0], [-0.1, 0], [0, 0.1], [0, -0.1]])

def bce(t, p):
    """Binary cross-entropy without the penalty and without Keras' clipping of the probabilities."""
    p = p.astype("float64")
    return float(np.mean(-(t * np.log(p) + (1 - t) * np.log1p(-p))))

for lam in LAMS:
    model = build(regularizers.L2(lam) if lam else None)
    h = model.fit(X, y, epochs=2000, validation_split=0.2, verbose=0).history
    p0 = model.predict(X, verbose=0).ravel()
    sens = np.mean([np.abs(model.predict(X + m, verbose=0).ravel() - p0) for m in moves])
    w = model.get_weights()[0].reshape(256)
    sweep_w[f"{lam:g}"] = w
    sweep_grid[f"{lam:g}"] = model.predict(coarse, verbose=0).ravel()
    sweep_rows.append({"lam": lam, "train_acc": h["accuracy"][-1], "val_acc": h["val_accuracy"][-1],
                       "train_loss_data": bce(y[:80], p0[:80]), "val_loss_data": bce(y[80:], p0[80:]),
                       "val_loss": h["val_loss"][-1], "largest_w": np.abs(w).max(), "sensitivity": sens})
    print(sweep_rows[-1], flush=True)
pd.DataFrame(sweep_rows).round(5).to_csv("data/lambda_sweep.csv", index=False)
pd.DataFrame(sweep_w).round(5).to_csv("data/lambda_sweep_weights.csv", index=False)
np.savez_compressed("data/lambda_sweep_grid.npz", **{k: np.round(v, 2).astype("float32") for k, v in sweep_grid.items()})